In [0]:
%run ./UTILS
%run ./CONFIG

dbutils.widgets.text("batch_control_key", "", "Batch Control Key")

BATCH_CONTROL_KEY = dbutils.widgets.get("batch_control_key").strip()
assert BATCH_CONTROL_KEY, "batch_control_key parameter is required"

print(f" Batch Control Key: {BATCH_CONTROL_KEY}")

# ===================================================================== #
# Read active config and extract silver-layer parameters                #
# ===================================================================== #
config_df = (
    spark.read.table(STREAMING_METADATA_TABLE)
    .filter(
        (F.col("batch_control_key") == BATCH_CONTROL_KEY) &
        (F.col("is_active") == True)
    )
)

assert config_df.count() > 0, (
    f"No active config found for '{BATCH_CONTROL_KEY}' in {STREAMING_METADATA_TABLE}"
)

config = config_df.collect()[0].asDict()

bronze_table_name = config["bronze_table_name"]
silver_table_name = config.get("silver_table_name")
silver_custom_notebook_url = (config.get("silver_custom_notebook_url") or "").strip()
partition_column = config.get("partition_column")
data_format = (config.get("data_format") or "JSON").strip().upper()
xml_row_tag = config.get("xml_row_tag", "row") or "row"
xsd_schema_path = (config.get("xsd_schema_path") or "").strip() or None
source_system = config["source_system"]

AUDIT_BATCH_KEY = f"{BATCH_CONTROL_KEY}_SILVER"

print(f" Bronze Table        : {bronze_table_name}")
print(f" Silver Table        : {silver_table_name}")
print(f" Data Format         : {data_format}")
print(f" XSD Schema Path     : {xsd_schema_path or '(none)'}")
print(f" Custom Notebook     : {silver_custom_notebook_url or '(direct write)'}")
print(f" Source System       : {source_system}")

if not silver_table_name:
    print("\n \u26a0 silver_table_name not configured. Nothing to process.")
    dbutils.notebook.exit(json.dumps({"status": "SKIPPED", "message": "No silver_table_name configured"}))


def _read_xsd_types(xsd_path):
    """Parse an XSD schema file and extract column-to-Spark type mappings.

    Reads an XSD file from a Unity Catalog Volume and maps XML Schema types
    to Spark SQL types. Used to apply proper data types from the source
    system's schema definition.

    Parameters
    ----------
    xsd_path : str
        UC Volumes path to the .xsd file.
        Example: /Volumes/{catalog_name}/framework/xsd_schemas/location.xsd

    Returns
    -------
    dict or None
        Mapping of {column_name: spark_type} for non-string columns.
        Returns None if the file cannot be read or parsed.
    """
    import xml.etree.ElementTree as ET

    XSD_TO_SPARK_TYPE = {
        "integer": "bigint",
        "int": "bigint",
        "long": "bigint",
        "short": "int",
        "decimal": "decimal(38,18)",
        "float": "double",
        "double": "double",
        "date": "date",
        "datetime": "timestamp",
        "string": "string",
        "token": "string",
        "normalizedstring": "string",
        "boolean": "boolean",
    }

    try:
        xsd_rows = spark.read.text(xsd_path).collect()
        xsd_content = "\n".join([row["value"] for row in xsd_rows])
        root = ET.fromstring(xsd_content)

        cast_map = {}
        for elem in root.iter("{http://www.w3.org/2001/XMLSchema}element"):
            col_name = elem.get("name")
            xsd_type = elem.get("type", "xs:string")
            if not col_name:
                continue
            local_type = xsd_type.split(":")[-1].lower().strip()
            spark_type = XSD_TO_SPARK_TYPE.get(local_type, "string")
            if spark_type != "string":
                cast_map[col_name] = spark_type

        if cast_map:
            print(f" \u2713 XSD schema parsed: {len(cast_map)} non-string type(s) from {xsd_path}")
            return cast_map
        else:
            print(f" \u26a0 XSD schema parsed but all columns are string type")
            return None

    except Exception as e:
        print(f" \u26a0 Could not read/parse XSD file at {xsd_path}: {e}")
        return None


def _get_xsd_string_fields(xsd_path):
    """Parse XSD and return a set of element names declared as string types.

    Used to override Spark's numeric type inference and preserve leading
    zeros in fields like barCode.

    Parameters
    ----------
    xsd_path : str
        UC Volumes path to the .xsd file.
    """
    import xml.etree.ElementTree as ET

    XSD_STRING_TYPES = {
        "string", "normalizedstring", "token", "nmtoken", "name", "ncname",
        "id", "idref", "entity", "anyuri", "language",
    }

    try:
        xsd_rows = spark.read.text(xsd_path).collect()
        xsd_content = "\n".join([row["value"] for row in xsd_rows])
        root = ET.fromstring(xsd_content)

        string_fields = set()
        for elem in root.iter("{http://www.w3.org/2001/XMLSchema}element"):
            name = elem.get("name")
            xsd_type = elem.get("type", "xs:string")
            if not name:
                continue
            local_type = xsd_type.split(":")[-1].lower().strip()
            if local_type in XSD_STRING_TYPES:
                string_fields.add(name)

        if string_fields:
            print(f" \u2713 XSD string fields: {len(string_fields)} field(s) declared as string")
            return string_fields
        return set()

    except Exception as e:
        print(f" \u26a0 Could not parse XSD for string fields: {e}")
        return set()


def _standardize_decimal_in_ddl(schema_ddl):
    """Standardize DECIMAL(p,s) \u2192 DECIMAL(38,18) in a DDL schema string.

    schema_of_xml infers decimal values with varying precision/scale.
    The framework standardizes all decimals to DECIMAL(38,18), matching
    the batch framework's decimal handling.

    Handles both top-level and nested DECIMAL types in the DDL string.

    Parameters
    ----------
    schema_ddl : str
        DDL schema string from schema_of_xml().

    Returns
    -------
    str
        DDL with all DECIMAL(p,s) replaced by DECIMAL(38,18).
    """
    import re as _re

    count = len(_re.findall(r'DECIMAL\(\d+,\s*\d+\)', schema_ddl, _re.IGNORECASE))
    standardized = _re.sub(
        r'DECIMAL\(\d+,\s*\d+\)',
        'DECIMAL(38,18)',
        schema_ddl,
        flags=_re.IGNORECASE
    )

    if count > 0:
        print(f" \u2713 Decimal standardization: {count} DECIMAL field(s) \u2192 DECIMAL(38,18)")
    return standardized


def _override_string_fields_in_ddl(schema_ddl, string_fields):
    """Override numeric types to STRING in DDL for XSD-declared string fields.

    Prevents Spark's type inference from converting barCode-like fields
    to BIGINT, preserving leading zeros and original string values.
    Handles nested fields at any depth in the DDL structure.

    Parameters
    ----------
    schema_ddl : str
        DDL schema string (potentially already decimal-standardized).
    string_fields : set
        Field names declared as string in XSD.

    Returns
    -------
    str
        DDL with matching numeric fields changed to STRING.
    """
    import re as _re

    override_count = 0
    for field_name in string_fields:
        new_ddl = _re.sub(
            rf'(?{_re.escape(field_name)}`?\s*:\s*)(BIGINT|INT|LONG|SHORT|DOUBLE|FLOAT|DECIMAL\([^)]*\))',
            rf'\1STRING',
            schema_ddl,
            flags=_re.IGNORECASE
        )

        if new_ddl != schema_ddl:
            override_count += 1
            schema_ddl = new_ddl

    if override_count > 0:
        print(f" \u2713 XSD string override: {override_count} numeric field(s) \u2192 STRING")
    return schema_ddl


def _apply_xsd_types(df, xsd_cast_map):
    """Apply XSD type declarations to DataFrame columns using try_cast.

    After from_xml parsing and flattening, some columns may still need
    type correction based on the XSD schema. Uses try_cast for safe
    conversion (returns NULL instead of error on edge cases).

    Parameters
    ----------
    df : DataFrame
        Flattened silver DataFrame.
    xsd_cast_map : dict
        Mapping of {column_name: spark_type} from _read_xsd_types().

    Returns
    -------
    DataFrame with XSD-declared types applied.
    """
    df_columns = set(df.columns)
    applicable_casts = {col: dtype for col, dtype in xsd_cast_map.items() if col in df_columns}

    if not applicable_casts:
        print(" \u26a0 XSD types found but no matching columns in DataFrame")
        return df

    # Apply all casts at once using withColumns (SCPAP004)
    cast_exprs = {
        col: F.expr(f"try_cast({col} as {dtype})")
        for col, dtype in applicable_casts.items()
    }
    df = df.withColumns(cast_exprs)

    type_counts = {}
    for dtype in applicable_casts.values():
        type_counts[dtype] = type_counts.get(dtype, 0) + 1
    summary = ", ".join(f"{count} {dtype}" for dtype, count in sorted(type_counts.items()))
    print(f" \u2713 Applied XSD types: cast {len(applicable_casts)} column(s) ({summary})")

    return df

In [0]:
import json
import traceback
from datetime import datetime
import xml.etree.ElementTree as ET

def _build_ddl_from_xsd(xsd_path, row_tag=None):
    """Build a complete Spark DDL schema string directly from an XSD file.

    Eliminates sample-based schema inference (schema_of_xml on first row).
    XSD is the single source of truth for both structure AND types.

    When row_tag is provided the function locates the matching
    <xs:element name="{row_tag}"> anywhere in the XSD (including
    nested definitions) and builds the DDL from *that* element's type.
    This prevents an extra nesting level when the XSD root is a wrapper
    (e.g. <resultSet>) and the bronze raw_xml_payload contains
    individual row elements (e.g. <resultSetRow>...</resultSetRow>).

    Handles:
      - Named complexTypes and type references (tns:TypeName)
      - xs:sequence with nested elements
      - maxOccurs="unbounded" -> ARRAY<...>
      - Inline complexTypes (nested structs)
      - All standard XSD simple types -> Spark types
      - Decimal -> DECIMAL(38,18) by default (framework standard)

    Parameters
    ----------
    xsd_path : str
        UC Volumes path to the .xsd file.
    row_tag : str, optional
        XML row tag name (e.g. 'resultSetRow'). When provided, the DDL
        is built from the element matching this name rather than the
        XSD's root element.

    Returns
    -------
    str or None
        DDL schema string for from_xml(), or None on failure.
    """
    XSD_NS = "{http://www.w3.org/2001/XMLSchema}"

    XSD_TYPE_MAP = {
        "string": "STRING",
        "int": "INT",
        "integer": "INT",
        "long": "BIGINT",
        "short": "SMALLINT",
        "decimal": "DECIMAL(38, 18)",
        "float": "DOUBLE",
        "double": "DOUBLE",
        "date": "DATE",
        "time": "STRING",  # Spark has no TIME type
        "datetime": "TIMESTAMP",
        "boolean": "BOOLEAN",
        "token": "STRING",
        "normalizedstring": "STRING",
    }

    try:
        # Collect all named complexTypes
        named_types = {}
        for ct in root.findall(f"{XSD_NS}complexType"):
            ct_name = ct.get("name")
            if ct_name:
                named_types[ct_name] = ct

        def _resolve_type(type_attr):
            """Resolve a type attribute (xs:string, tns:PlannedReceiptType, etc.)."""
            if not type_attr:
                return None
            local = type_attr.split(":")[-1]
            # Check simple types first
            spark_type = XSD_TYPE_MAP.get(local.lower())
            if spark_type:
                return spark_type
            # Check named complexTypes
            if local in named_types:
                return _build_struct(named_types[local])
            return "STRING"  # Fallback for unknown types

        def _build_struct(complex_type_elem):
            """Build STRUCT<...> DDL from a complexType element."""
            seq = complex_type_elem.find(f"{XSD_NS}sequence")
            if seq is None:
                return "STRING"

            fields = []
            for elem in seq.findall(f"{XSD_NS}element"):
                field_name = elem.get("name")
                if not field_name:
                    continue

                max_occurs = elem.get("maxOccurs", "1")
                is_array = max_occurs == "unbounded"

                # Check for inline complexType (nested struct)
                inline_ct = elem.find(f"{XSD_NS}complexType")
                if inline_ct is not None:
                    field_type = _build_struct(inline_ct)
                else:
                    type_attr = elem.get("type")
                    field_type = _resolve_type(type_attr) or "STRING"

                if is_array:
                    field_type = f"ARRAY<{field_type}>"

                fields.append(f"{field_name}: {field_type}")

            return "STRUCT<" + ", ".join(fields) + ">"

        # Find the element matching row_tag (row-level), or fall back to
        # the XSD root element. Searching all elements (iter) handles
        # XSDs where the row tag is nested inside a wrapper element.
        target_elem = None
        if row_tag:
            for elem in root.iter(f"{XSD_NS}element"):
                if elem.get("name") == row_tag:
                    target_elem = elem
                    break
            if target_elem is not None:
                print(f"✓ XSD: matched row_tag '{row_tag}' element")
            else:
                print(f"⚠️ XSD: row_tag '{row_tag}' not found - falling back to root element")

        if target_elem is None:
            target_elem = root.find(f"{XSD_NS}element")

        if target_elem is None:
            print(f"⚠️ No root element found in XSD")
            return None

        root_elem = target_elem

        root_type_attr = root_elem.get("type")
        inline_ct = root_elem.find(f"{XSD_NS}complexType")

        if inline_ct is not None:
            ddl = _build_struct(inline_ct)
        elif root_type_attr:
            ddl = _resolve_type(root_type_attr)
        else:
            ddl = "STRING"

        field_count = ddl.count(":") if ddl else 0
        print(f"✓ XSD DDL built: {field_count} field(s) from {xsd_path}")
        return ddl

    except Exception as e:
        print(f"⚠️ Could not build DDL from XSD at {xsd_path}: {e}")
        return None


print("XSD type parsing and decimal standardization helpers loaded.")


# ========================================================================= #
# Read unprocessed records from bronze table
# Watermark: records inserted after last successful silver audit
# ========================================================================= #
job_start = datetime.now()
watermark_query = (
    spark.read.table(AUDIT_TABLE)
    .filter(
        (F.col("BATCH_CTRL_KEY") == AUDIT_BATCH_KEY) &
        (F.col("JOB_STATUS") == "SUCCESS")
    )
    .agg(F.max("JOB_END_TIMESTAMP").alias("last_success"))
)

NOTEBOOK_PATH = _notebook_path
print(NOTEBOOK_PATH)
try:
    watermark_ts = watermark_query.collect()[0]["last_success"]
except Exception as e:
    watermark_ts = None
    print(f"⚠️ Watermark lookup failed (defaulting to full read): {e}")

if watermark_ts:
    print(f"Watermark: {watermark_ts} (reading records after this timestamp)")
    bronze_df = (
        spark.read.table(bronze_table_name)
        .filter(F.col("_created_ts") > F.lit(watermark_ts))
    )
else:
    print("No watermark found - reading all bronze records")
    bronze_df = spark.read.table(bronze_table_name)

bronze_columns = set(bronze_df.columns)
bronze_passthrough_cols = [
    c for c in [
        "_source_system_code",
        "_division_code",
        "_data_domain",
        "_record_guid"
    ]
    if c in bronze_columns
]

if bronze_df.limit(1).count() == 0:
    print("No new records to process.")
    job_end = datetime.now()
    audit_rec = build_audit_record(
        AUDIT_BATCH_KEY, source_system, BATCH_CONTROL_KEY,
        0, "SUCCESS", job_start, job_end, NOTEBOOK_PATH=NOTEBOOK_PATH,
        extra_job_stats={"message": "No new records"}
    )
    # Preserve existing watermark - don't advance on empty batch
    audit_rec["JOB_END_TIMESTAMP"] = watermark_ts or job_start
    insert_audit_record(audit_rec, AUDIT_TABLE)
    dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "records_processed": 0, "message": "No new records"}))

try:
    _max_bronze_ts = bronze_df.agg(F.max("_created_ts")).collect()[0][0]
except Exception as e:
    _max_bronze_ts = None
    print(f"⚠️ Max bronze timestamp lookup failed: {e}")

print("Bronze records found - proceeding to parse")

if data_format == "JSON":
    # Infer JSON schema from a single sample (1 row to driver, not all)
    sample_json = (
        bronze_df
        .select("raw_xml_payload")
        .filter(F.col("raw_xml_payload").isNotNull())
        .limit(1)
        .collect()[0][0]
    )

    # schema_of_json returns a DDL schema string - no RDD needed
    schema_ddl = (
        spark.createDataFrame([(sample_json,)], ["j"])
        .select(F.schema_of_json(F.col("j")).alias("s"))
        .collect()[0][0]
    )

    parsed_df = bronze_df.withColumn("parsed", F.from_json(F.col("raw_xml_payload"), schema_ddl))
    silver_df = _flatten_nested_df(
        parsed_df.select("parsed.*", *bronze_passthrough_cols)
    )

    # Add audit columns (matching XML and fallback paths)
    silver_df = (
        silver_df
        .withColumn("_created_ts", F.current_timestamp())
        .withColumn("_updated_ts", F.current_timestamp())
        .withColumn("_pipeline_name", F.lit(AUDIT_BATCH_KEY))
        .withColumn("_pipeline_run_id", F.lit(_pipeline_run_id))
        .withColumn("partition_date", F.to_date(F.current_timestamp()))
    )

elif data_format == "XML":
    # ---- XSD-based schema & type resolution ---- #
    # Priority 1: Build DDL directly from XSD (no sample dependency)
    # Priority 2: Fallback to schema_of_xml on a sample row (when no XSD)
    xsd_cast_map = None
    typed_schema_ddl = None

    if xsd_schema_path:
        typed_schema_ddl = _build_ddl_from_xsd(xsd_schema_path, row_tag=xml_row_tag)
        xsd_cast_map = _read_xsd_types(xsd_schema_path)

    if not typed_schema_ddl:
        # Fallback: infer from sample (when no XSD or XSD parsing failed)
        print("⚠️ No XSD DDL - falling back to schema_of_xml (single-sample inference)")
        sample_xml = (
            bronze_df
            .select("raw_xml_payload")
            .where(F.col("raw_xml_payload").isNotNull())
            .orderBy(F.length("raw_xml_payload").desc())
            .first()[0]
        )

        xml_schema_ddl = (
            spark.range(1)
            .select(F.schema_of_xml(F.lit(sample_xml)).alias("schema"))
            .first()["schema"]
        )

        # Apply decimal standardization and string overrides on inferred DDL
        typed_schema_ddl = _standardize_decimal_in_ddl(xml_schema_ddl)
        xsd_string_fields = _get_xsd_string_fields(xsd_schema_path) if xsd_schema_path else set()
        if xsd_string_fields:
            typed_schema_ddl = _override_string_fields_in_ddl(typed_schema_ddl, xsd_string_fields)

    parsed_df = bronze_df.withColumn(
        "parsed", F.from_xml(F.col("raw_xml_payload"), typed_schema_ddl)
    )

    silver_df = _flatten_nested_df(
        parsed_df.select("parsed.*", *bronze_passthrough_cols).drop("_created_ts")
    )

    # Apply XSD type casting to flattened columns (safety net after flatten)
    if xsd_cast_map:
        silver_df = _apply_xsd_types(silver_df, xsd_cast_map)

    # Add audit columns
    silver_df = (
        silver_df
        .withColumn("_created_ts", F.current_timestamp())
        .withColumn("_updated_ts", F.current_timestamp())
        .withColumn("_pipeline_name", F.lit(AUDIT_BATCH_KEY))
        .withColumn("_pipeline_run_id", F.lit(_pipeline_run_id))
        .withColumn("partition_date", F.to_date(F.current_timestamp()))
    )

else:
    # Pass through raw payload for unsupported formats
    silver_df = (
        bronze_df.select("raw_xml_payload", *bronze_passthrough_cols)
        .withColumns({
            "_created_ts": F.current_timestamp(),
            "_updated_ts": F.current_timestamp(),
            "_pipeline_name": F.lit(AUDIT_BATCH_KEY),
            "_pipeline_run_id": F.lit(_pipeline_run_id),
            "partition_date": F.to_date(F.current_timestamp()),
        })
    )
    print(f"⚠️ Unsupported data_format '{data_format}' — passing raw message_body")

print(f"Parsed silver schema: {len(silver_df.columns)} columns")

try:
    if silver_custom_notebook_url:
        # ---- Delegate to custom silver notebook ----
        print(f"Delegating to custom notebook: {silver_custom_notebook_url}")

        # Write parsed data to a temp view for the custom notebook to read
        temp_view = f"bronze_silver{BATCH_CONTROL_KEY.lower().replace('-', '_')}"
        silver_df.createOrReplaceGlobalTempView(temp_view)

        result = dbutils.notebook.run(
            silver_custom_notebook_url,
            timeout_seconds=3600,
            arguments={
                "batch_control_key": BATCH_CONTROL_KEY,
                "bronze_table_name": bronze_table_name,
                "silver_table_name": silver_table_name,
                "source_system": source_system,
                "partition_column": partition_column or "",
                "temp_view": temp_view,
            },
        )
        print(f"✓ Custom notebook completed: {result}")

        try:
            result_dict = json.loads(result)
            records_processed = result_dict.get("records_processed", 0)
        except (json.JSONDecodeError, TypeError):
            records_processed = 0
            print(f"⚠️ Could not parse custom notebook result as JSON (records_processed defaults to 0): {result}")

        # Clean up global temp view to prevent accumulation in continuous orchestration
        spark.catalog.dropGlobalTempView(temp_view)

    else:
        # ---- Direct write via common method (no custom transformations) ----
        records_processed = write_to_target_table(
            silver_df,
            silver_table_name,
            BATCH_CONTROL_KEY,
            partition_column=partition_column,
        )

    status = "SUCCESS"
    error_msg = None

except Exception as e:
    status = "FAILED"
    error_msg = str(e)
    records_processed = 0
    print(f"❌ Silver processing FAILED: {error_msg}")
    traceback.print_exc()


# ========================================================================= #
# Audit record + notebook exit
# JOB_END_TIMESTAMP stores max(insert_timestamp) from the processed
# batch - NOT datetime.now(). This prevents data loss when real-time
# ========================================================================= #
job_end = datetime.now()

audit_rec = build_audit_record(
    AUDIT_BATCH_KEY, source_system, BATCH_CONTROL_KEY,
    records_processed, status, job_start, job_end,
    error_message=error_msg, NOTEBOOK_PATH=NOTEBOOK_PATH,
    extra_job_stats={
        "bronze_table": bronze_table_name,
        "silver_table": silver_table_name,
        "data_format": data_format,
        "records_processed": records_processed,
        "actual_job_end": str(job_end),
    },
)

# Override JOB_END_TIMESTAMP with max data timestamp for watermark safety
if status == "SUCCESS" and _max_bronze_ts:
    audit_rec["JOB_END_TIMESTAMP"] = _max_bronze_ts

insert_audit_record(audit_rec, AUDIT_TABLE)

# Fail-fast: if any child notebook failed, raise so the orchestrator
# stops the pipeline instead of silently continuing to the next stage.
if status == "FAILED":
    raise Exception(
        f"BRONZE_SILVER failed for {BATCH_CONTROL_KEY}: {error_msg}"
    )

exit_with_reconciliation(
    BATCH_CONTROL_KEY, records_processed,
    status=status, error_message=error_msg or "",
)